<center><h1>Steven-Schneider_Isla_HW7</h1></center>

Name: Isla Steven-Schneider
<br>
Github Username: islastevenschneider
<br>
USC ID: 1985-4980-09

## 1. Multi-class and Multi-Label Classification Using Support Vector Machines

Import packages

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as SmotePipeline
from imblearn.over_sampling import SMOTE
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score

np.random.seed(1113)

### (a) Download the Anuran Calls (MFCCs) Data Set

In [2]:
df = pd.read_csv("../data/Frogs_MFCCs.csv")

train_set, test_set = train_test_split(df, test_size=0.3, random_state=1113)

print(df.shape, train_set.shape, test_set.shape)

(7195, 26) (5036, 26) (2159, 26)


### (b) Train a classifier for each label

#### (i) Research

For exact match I use the Exact Match Ratio: the fraction of instances where all three labels (family, genus, species) are predicted correctly. This is a harsh measure, since two of three labels right counts the same as none right (Sorower, 2010).

Hamming loss gives partial credit (Schapire & Singer, 2000). It is the fraction of the three labels predicted wrong, averaged over instances, so lower is better. The Hamming score is 1 − Hamming loss, the fraction of labels predicted correctly, in the same spirit as the partial-credit accuracy of Godbole and Sarawagi (2004). I report all three for each classifier.

References:

Godbole, S., & Sarawagi, S. (2004). Discriminative methods for multi-labeled classification. In *Proceedings of the 8th Pacific-Asia Conference on Knowledge Discovery and Data Mining (PAKDD 2004)* (pp. 22–30).

Schapire, R. E., & Singer, Y. (2000). BoosTexter: A boosting-based system for text categorization. *Machine Learning, 39*(2/3), 135–168.

Sorower, M. S. (2010). *A literature survey on algorithms for multi-label learning*. Department of Computer Science, Oregon State University.

#### (ii) Train a SVM for each of the labels

In [3]:
features = df.columns[:22]
labels = ["Family", "Genus", "Species"]
x_train = train_set[features]
x_test = test_set[features]

sigmas = np.arange(0.2, 2.2, 0.2)
search_grid = {"estimator__C": 10.0 ** np.arange(-1, 5),
               "estimator__gamma": 1 / (2 * sigmas**2)}

gaussian_pred = pd.DataFrame(index=test_set.index)
for label in labels:
    gaussian_search = GridSearchCV(OneVsRestClassifier(SVC(kernel="rbf")),
                                   search_grid, cv=10, n_jobs=-1)
    gaussian_search.fit(x_train, train_set[label])
    gaussian_pred[label] = gaussian_search.predict(x_test)
    C = gaussian_search.best_params_["estimator__C"]
    sigma = np.sqrt(1 / (2 * gaussian_search.best_params_["estimator__gamma"]))
    print(label, "C =", C, "sigma =", round(sigma, 1),
          "test accuracy =", round(gaussian_search.score(x_test, test_set[label]), 4))

correct = gaussian_pred == test_set[labels]
exact_match = correct.all(axis=1).mean()
hamming_loss = (1 - correct).mean().mean()
hamming_score = 1 - hamming_loss
print("Exact match:", round(exact_match, 4))
print("Hamming loss:", round(hamming_loss, 4))
print("Hamming score:", round(hamming_score, 4))

Family C = 100.0 sigma = 0.4 test accuracy = 0.9926
Genus C = 10.0 sigma = 0.6 test accuracy = 0.9893
Species C = 100.0 sigma = 0.4 test accuracy = 0.9903
Exact match: 0.9866
Hamming loss: 0.0093
Hamming score: 0.9907


#### (iii) Repeat 1(b)ii with L1-penalized SVMs

In [4]:
linear_grid = {"svm__C": 10.0 ** np.arange(-3, 4)}

linear_pred = pd.DataFrame(index=test_set.index)
for label in labels:
    linear_search = GridSearchCV(Pipeline([("scale", StandardScaler()),
                                           ("svm", LinearSVC(penalty="l1", dual=False, max_iter=10000))]),
                                 linear_grid, cv=10, n_jobs=-1)
    linear_search.fit(x_train, train_set[label])
    linear_pred[label] = linear_search.predict(x_test)
    print(label, "C =", linear_search.best_params_["svm__C"],
          "test accuracy =", round(linear_search.score(x_test, test_set[label]), 4))

correct = linear_pred == test_set[labels]
exact_match = correct.all(axis=1).mean()
hamming_loss = (1 - correct).mean().mean()
hamming_score = 1 - hamming_loss
print("Exact match:", round(exact_match, 4))
print("Hamming loss:", round(hamming_loss, 4))
print("Hamming score:", round(hamming_score, 4))

Family C = 1.0 test accuracy = 0.9296
Genus C = 100.0 test accuracy = 0.9458
Species C = 10.0 test accuracy = 0.9491
Exact match: 0.9106
Hamming loss: 0.0585
Hamming score: 0.9415


#### (iv) Repeat 1(b)iii by using SMOTE or any other method for imbalance

In [5]:
smote_grid = {"svm__C": 10.0 ** np.arange(-3, 4)}

smote_pred = pd.DataFrame(index=test_set.index)
for label in labels:
    smote_search = GridSearchCV(SmotePipeline([("scale", StandardScaler()),
                                               ("smote", SMOTE(random_state=1113)),
                                               ("svm", LinearSVC(penalty="l1", dual=False, max_iter=10000))]),
                                smote_grid, cv=10, n_jobs=-1)
    smote_search.fit(x_train, train_set[label])
    smote_pred[label] = smote_search.predict(x_test)
    print(label, "C =", smote_search.best_params_["svm__C"],
          "test accuracy =", round(smote_search.score(x_test, test_set[label]), 4))

correct = smote_pred == test_set[labels]
exact_match = correct.all(axis=1).mean()
hamming_loss = (1 - correct).mean().mean()
hamming_score = 1 - hamming_loss
print("Exact match:", round(exact_match, 4))
print("Hamming loss:", round(hamming_loss, 4))
print("Hamming score:", round(hamming_score, 4))

Family C = 1000.0 test accuracy = 0.9115


/Users/isla/miniconda3/envs/dsci552/lib/python3.12/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/isla/miniconda3/envs/dsci552/lib/python3.12/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Genus C = 10.0 test accuracy = 0.9162


/Users/isla/miniconda3/envs/dsci552/lib/python3.12/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/isla/miniconda3/envs/dsci552/lib/python3.12/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/isla/miniconda3/envs/dsci552/lib/python3.12/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/isla/miniconda3/envs/dsci552/lib/python3.12/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/isla/miniconda3/envs/dsci552/lib/python3.12/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/isla/miniconda3/envs/ds

Species C = 1.0 test accuracy = 0.9486
Exact match: 0.856
Hamming loss: 0.0746
Hamming score: 0.9254


Of the three classifiers I trained, the Gaussian SVM was the best on all three measures (exact match 0.987, Hamming loss 0.009, Hamming score 0.991) and did much better than the other two. This makes sense because it has a nonlinear boundary. Frogs in the same class can have different-sounding calls, so the classes can't be split with one flat boundary.

The L1 linear SVM was not as good (exact match 0.911, Hamming loss 0.059) because it can only separate classes with hyperplanes. 

For the third model I added SMOTE to the linear SVM, and this made it worse on all three measures (exact match 0.856, Hamming loss 0.075). SMOTE evens out the classes to help the rare ones. Since exact match and Hamming loss count every frog equally, the large classes dominate, so shifting weight towards the rare classes means more of the common ones get misclassified. 

Exact match is always below the Hamming score, and the gap grows as the classifier gets worse (0.004, 0.031, 0.069). Each label is predicted separately, so a frog can be right on some labels and wrong on others, and that happens more often with the weaker classifiers.


## 2. K-Means Clustering on a Multi-Class and Multi-Label Data Set

### (a) Use k-means clustering

In [6]:
x_all = df[features]

def pick_clusters(x):
    best_ch = 0
    for k in range(2, 51):
        clusters = KMeans(n_clusters=k).fit_predict(x)
        ch = calinski_harabasz_score(x, clusters)
        if ch > best_ch:
            best_ch = ch
            best_clusters = clusters
    return best_clusters

clusters = pick_clusters(x_all)
print("Using the CH index for k from 2 to 50 (not defined at k = 1), k =", clusters.max() + 1)

Using the CH index for k from 2 to 50 (not defined at k = 1), k = 2


### (b) Determine which family is the majority

In [7]:
def most_common_labels(clusters):
    majority = pd.DataFrame(index=range(clusters.max() + 1), columns=labels)
    for c in majority.index:
        for label in labels:
            majority.loc[c, label] = df[label][clusters == c].value_counts().idxmax()
    return majority

most_common_labels(clusters)

,Family,Genus,Species
0,Leptodactylidae,Adenomera,AdenomeraHylaedactylus
1,Hylidae,Hypsiboas,HypsiboasCordobae


### (c) Calculate the average Hamming distance, Hamming score, and Hamming loss

In [8]:
num_runs = 50
hamming_distances = []
for run in range(num_runs):
    clusters = pick_clusters(x_all)
    majority = most_common_labels(clusters)
    wrong = majority.loc[clusters].values != df[labels].values
    hamming_distances.append(wrong.sum(axis=1).mean())

average_distance = np.mean(hamming_distances)
print("Average Hamming distance:", round(average_distance, 4))
print("Standard deviation:", round(np.std(hamming_distances, ddof=1), 4), "*")
print("Hamming loss:", round(average_distance / 3, 4))
print("Hamming score:", round(1 - average_distance / 3, 4))
print("* CH picked k = 2 in every run and k-means found the same two clusters each time, so all 50 distances are equal.")

Average Hamming distance: 0.8956
Standard deviation: 0.0 *
Hamming loss: 0.2985
Hamming score: 0.7015
* CH picked k = 2 in every run and k-means found the same two clusters each time, so all 50 distances are equal.


## 3. ISLR 12.6.2

### (a) Dendrogram with Complete Linkage

Let $d_{ij}$ be the dissimilarity between observations $i$ and $j$:

$$D = \begin{pmatrix} & 0.3 & 0.4 & 0.7 \\ 0.3 & & 0.5 & 0.8 \\ 0.4 & 0.5 & & 0.45 \\ 0.7 & 0.8 & 0.45 & \end{pmatrix}$$

- $\min_{i \neq j} d_{ij} = d_{12} = 0.3$, so merge $\{1,2\}$ at height $0.3$.
- $d(\{1,2\}, 3) = \max(0.4, 0.5) = 0.5$
- $d(\{1,2\}, 4) = \max(0.7, 0.8) = 0.8$
- $d(3, 4) = 0.45$ is smaller than both, so $\{3,4\}$ merge next at height $0.45$.
- $d(\{1,2\}, \{3,4\}) = \max(0.4, 0.7, 0.5, 0.8) = 0.8$, so the last merge is at height $0.8$.

```
0.8         ┌------------------┐
            |                  |
0.45        |              ┌---|---┐
0.3     ┌---|---┐          |       |
        1       2          3       4
```

### (b) Dendrogram with Single Linkage

- The first merge doesn't depend on the linkage, so $\{1,2\}$ merge at height $0.3$ again.
- $d(\{1,2\}, 3) = \min(0.4, 0.5) = 0.4$
- $d(\{1,2\}, 4) = \min(0.7, 0.8) = 0.7$
- $d(3, 4) = 0.45$
- The smallest is $0.4$, so merge $\{1,2\}$ with $3$ at height $0.4$.
- $d(\{1,2,3\}, 4) = \min(0.7, 0.8, 0.45) = 0.45$, so $4$ joins at height $0.45$.

```
0.45           ┌---------------┐
0.4       _____|______         |
         |            |        |
0.3   ┌--|--┐         |        |  
      1     2         3        4
```

### (c) Cutting Dendrogram 1

If we cut the dendrogram from part a) to get 2 clusters, so at any height in $(0.45, 0.8)$, this gives the clusters $\{1, 2\}$ and $\{3, 4\}$ because only the last merge is above the cut.


### (d) Cutting Dendrogram 2

If we cut the dendrogram from part b) to get 2 clusters, so at any height in $(0.4, 0.45)$, this gives the clusters $\{1, 2, 3\}$ and $\{4\}$, also because only the last merge is above the cut.

### (e) Swapping Clusters

When the leaves are swapped inside the lower merges, the result is the same dendrogram. This shows that the horizontal order doesn't matter, only which clusters merge and at what height.

```
0.8         ┌------------------┐
            |                  |
0.45        |              ┌---|---┐
0.3     ┌---|---┐          |       |
        2       1          4       3
```

